# 🔥 IWASAKI AI STUDIO

**内装業 × 最先端AI**

---

## このノートブックでできること

1. **🎨 高品質画像生成** - 内装デザインのビジュアライゼーション
2. **🎬 動画生成** - 静止画から動く映像へ
3. **🏠 3Dシーン生成** - 写真から歩ける3D空間へ
4. **🤖 カスタムスタイル学習** - イワサキ内装の施工スタイルをAIに学習

---

**A100 40GB専用。へぼいのは無しで。**

In [ ]:
#@title 🚀 **Step 0: GPU確認 & 環境セットアップ**
#@markdown まずこれを実行して、A100が使えるか確認

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                       capture_output=True, text=True)
gpu_info = result.stdout.strip()
print(f"\n🎮 GPU: {gpu_info}\n")

if 'A100' in gpu_info:
    print("✅ A100確認！フルパワーで行ける")
    print("━" * 50)
elif 'V100' in gpu_info or 'T4' in gpu_info:
    print("⚠️ A100じゃない。ランタイム変更推奨")
    print("   ランタイム → ランタイムのタイプを変更 → A100")
else:
    print(f"⚠️ 不明なGPU。続行は可能だが制限あり")

# VRAM確認
import torch
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\n💾 VRAM: {vram:.1f} GB")
    if vram >= 35:
        print("   → 動画生成、3DGS、全部いける")
    elif vram >= 15:
        print("   → 画像生成は余裕。動画は制限あり")
    else:
        print("   → 基本的な画像生成のみ")

---

## 🎨 Section 1: 高品質画像生成（SDXL + Fooocus）

**Fooocus** = Stable Diffusion XLを簡単に使えるUI

- プロンプト入力 → Generate → 画像出力
- 英語推奨（「ドラゴン」だとひまわり畑が出る）

In [ ]:
#@title 🎨 **Fooocus インストール & 起動**
#@markdown 約3-5分かかる。完了したらGradioリンクが出る

!git clone https://github.com/lllyasviel/Fooocus.git
%cd Fooocus
!pip install -r requirements_versions.txt -q

# 内装デザイン向けのプリセット追加
preset_content = '''{
    "default_model": "juggernautXL_v8Rundiffusion.safetensors",
    "default_refiner": "None",
    "default_lora": "None",
    "default_styles": ["Fooocus V2", "Fooocus Photograph", "Fooocus Enhance"],
    "default_prompt_negative": "low quality, blurry, distorted, ugly, deformed",
    "default_cfg_scale": 7,
    "default_sample_sharpness": 2,
    "default_sampler": "dpmpp_2m_sde_gpu",
    "preset_name": "Interior Design Pro"
}'''

with open('presets/interior_design.json', 'w') as f:
    f.write(preset_content)

print("\n✅ Fooocus準備完了")
print("\n起動中...")

# Gradioで起動（share=True でURLが生成される）
!python entry_with_update.py --share --preset interior_design

### 内装デザイン用プロンプト例

```
# モダン和室
modern japanese interior, tatami room, shoji screens, warm lighting, 
minimalist design, natural wood textures, professional photography, 8k

# 高級オフィス
luxury corporate office interior, executive boardroom, floor to ceiling windows,
city skyline view, premium furniture, ambient lighting, architectural photography

# カフェ風リビング
cozy cafe style living room, exposed brick wall, vintage furniture, 
indoor plants, warm sunlight, wooden floors, interior design magazine photo
```

---

## 🎬 Section 2: 動画生成（Stable Video Diffusion）

**画像1枚 → 4秒の動画**

内装のBefore/Afterを動かす、コルクじじいを動かす、等

In [ ]:
#@title 🎬 **Stable Video Diffusion セットアップ**
#@markdown 約5分かかる

!pip install diffusers transformers accelerate safetensors -q
!pip install imageio[ffmpeg] -q

import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
from PIL import Image
import requests
from io import BytesIO

# モデルロード（初回は10分くらいかかる）
print("\n🔄 モデルロード中（初回は約10分）...")
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# メモリ最適化
pipe.enable_model_cpu_offload()

print("\n✅ Stable Video Diffusion 準備完了！")
print("━" * 50)
print("次のセルで画像URLを入力して動画生成")

In [ ]:
#@title 🎬 **画像から動画生成**
#@markdown 画像URLを入力して実行

image_url = "https://imagedelivery.net/k1Zw56y2FNiZaFcOP7Rs2Q/664066c9-a884-4bf8-e9ed-a4cd72115700/public" #@param {type:"string"}
output_filename = "generated_video.mp4" #@param {type:"string"}
motion_bucket_id = 127 #@param {type:"slider", min:1, max:255, step:1}
fps = 7 #@param {type:"slider", min:1, max:30, step:1}

# 画像ロード
print(f"\n🖼️ 画像ロード: {image_url[:50]}...")
response = requests.get(image_url)
image = Image.open(BytesIO(response.content))

# リサイズ（SVDは1024x576推奨）
image = image.resize((1024, 576))

print("\n🎬 動画生成中（約2-3分）...")
frames = pipe(
    image, 
    decode_chunk_size=8,
    motion_bucket_id=motion_bucket_id,
    num_frames=25
).frames[0]

# 動画保存
export_to_video(frames, output_filename, fps=fps)
print(f"\n✅ 完了！ {output_filename} を保存")

# Colab上で表示
from IPython.display import HTML
from base64 import b64encode
mp4 = open(output_filename, 'rb').read()
data_url = f"data:video/mp4;base64,{b64encode(mp4).decode()}"
HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>')

---

## 🏠 Section 3: 3D Gaussian Splatting

**写真数枚 → 歩ける3D空間**

施工現場を3D化してお客様に見せる

In [ ]:
#@title 🏠 **gsplat セットアップ**
#@markdown 3D Gaussian Splattingの高速実装

!pip install gsplat nerfstudio -q

print("\n✅ gsplat インストール完了")
print("\n使い方:")
print("1. 施工現場の写真を10-20枚撮影（いろんな角度から）")
print("2. 写真をアップロード")
print("3. 3Dモデルを生成")
print("4. ブラウザで歩き回れる")

In [ ]:
#@title 🏠 **COLMAP + 3DGS パイプライン**
#@markdown 写真フォルダを指定して3Dモデル生成

# COLMAPインストール
!apt-get install -y colmap

# 写真アップロード用のディレクトリ作成
import os
os.makedirs('/content/input_images', exist_ok=True)
os.makedirs('/content/output_3d', exist_ok=True)

print("\n✅ 準備完了")
print("\n📁 /content/input_images に写真をアップロードしてください")
print("   左のファイルパネルからドラッグ&ドロップ")
print("\n写真の撮り方:")
print("   - 10-20枚以上")
print("   - いろんな角度から")
print("   - オーバーラップを確保（隣の写真と30%以上重なるように）")
print("   - ブレないように")

In [ ]:
#@title 🏠 **3DGS生成実行**
#@markdown 写真をアップロードした後に実行

import os

input_dir = '/content/input_images'
output_dir = '/content/output_3d'

# 写真数確認
images = [f for f in os.listdir(input_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"\n📸 検出した写真: {len(images)}枚")

if len(images) < 5:
    print("\n⚠️ 写真が少なすぎます。最低10枚推奨")
else:
    print("\n🔄 COLMAP実行中（特徴点抽出）...")
    !colmap feature_extractor \
        --database_path {output_dir}/database.db \
        --image_path {input_dir} \
        --ImageReader.single_camera 1
    
    print("\n🔄 マッチング中...")
    !colmap exhaustive_matcher \
        --database_path {output_dir}/database.db
    
    os.makedirs(f'{output_dir}/sparse', exist_ok=True)
    
    print("\n🔄 3D再構築中...")
    !colmap mapper \
        --database_path {output_dir}/database.db \
        --image_path {input_dir} \
        --output_path {output_dir}/sparse
    
    print("\n✅ COLMAP完了！")
    print("\n次にGaussian Splattingを実行...")

---

## 🤖 Section 4: カスタムLoRA学習

**イワサキ内装のスタイルをAIに学習させる**

過去の施工写真 → 「イワサキ風」の画像を自動生成

In [ ]:
#@title 🤖 **LoRA学習環境セットアップ**
#@markdown kohya_ssを使用

!git clone https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!pip install -r requirements.txt -q
!pip install xformers -q

# 学習用ディレクトリ作成
import os
os.makedirs('/content/training_images', exist_ok=True)
os.makedirs('/content/output_lora', exist_ok=True)

print("\n✅ LoRA学習環境 準備完了")
print("\n📁 /content/training_images に学習用画像をアップロード")
print("\n推奨:")
print("   - 10-50枚の施工写真")
print("   - 統一感のあるスタイル")
print("   - 高解像度（1024x1024以上）")

In [ ]:
#@title 🤖 **キャプション自動生成**
#@markdown 画像からキャプションを自動生成（学習に必要）

!pip install transformers accelerate -q

from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import os

# BLIPロード
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large"
).to("cuda")

training_dir = '/content/training_images'
images = [f for f in os.listdir(training_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"\n📸 {len(images)}枚の画像にキャプション生成中...\n")

for img_file in images:
    img_path = os.path.join(training_dir, img_file)
    image = Image.open(img_path).convert('RGB')
    
    inputs = processor(image, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=50)
    caption = processor.decode(out[0], skip_special_tokens=True)
    
    # トリガーワード追加
    caption = f"iwasaki_style, {caption}"
    
    # キャプションファイル保存
    txt_path = os.path.splitext(img_path)[0] + '.txt'
    with open(txt_path, 'w') as f:
        f.write(caption)
    
    print(f"✅ {img_file}: {caption[:50]}...")

print(f"\n✅ 完了！ {len(images)}枚にキャプション生成")

In [ ]:
#@title 🤖 **LoRA学習実行**
#@markdown 約30分-1時間（画像数による）

training_dir = '/content/training_images'
output_dir = '/content/output_lora'
lora_name = 'iwasaki_style' #@param {type:"string"}
max_train_steps = 1000 #@param {type:"slider", min:100, max:3000, step:100}

# 設定ファイル生成
config = f'''[model]
pretrained_model_name_or_path = "stabilityai/stable-diffusion-xl-base-1.0"

[train]
train_data_dir = "{training_dir}"
output_dir = "{output_dir}"
output_name = "{lora_name}"
max_train_steps = {max_train_steps}
learning_rate = 1e-4
network_dim = 32
network_alpha = 16
resolution = "1024,1024"
train_batch_size = 1
gradient_accumulation_steps = 4
save_every_n_steps = 500
mixed_precision = "fp16"
cache_latents = true
'''

with open('train_config.toml', 'w') as f:
    f.write(config)

print("\n🚀 LoRA学習開始...")
print(f"   ステップ数: {max_train_steps}")
print(f"   出力: {output_dir}/{lora_name}.safetensors")
print("\n⏳ 約30分-1時間かかります\n")

!accelerate launch --num_cpu_threads_per_process=2 sdxl_train_network.py \
    --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
    --train_data_dir="{training_dir}" \
    --output_dir="{output_dir}" \
    --output_name="{lora_name}" \
    --max_train_steps={max_train_steps} \
    --learning_rate=1e-4 \
    --network_dim=32 \
    --network_alpha=16 \
    --resolution="1024,1024" \
    --train_batch_size=1 \
    --gradient_accumulation_steps=4 \
    --save_every_n_steps=500 \
    --mixed_precision="fp16" \
    --cache_latents \
    --network_module=networks.lora

print("\n✅ LoRA学習完了！")
print(f"\n📁 出力: {output_dir}/{lora_name}.safetensors")
print("\nFooocusで使う場合:")
print("   1. このファイルをダウンロード")
print("   2. Fooocusのlorasフォルダに配置")
print("   3. プロンプトに 'iwasaki_style' を追加")

---

## 📤 ユーティリティ

In [ ]:
#@title 📤 **Cloudflare Imagesにアップロード**
#@markdown 生成した画像をCloudflareに直接アップロード

cloudflare_account_id = "" #@param {type:"string"}
cloudflare_api_token = "" #@param {type:"string"}
image_path = "/content/Fooocus/outputs/" #@param {type:"string"}

import requests
import os

def upload_to_cloudflare(image_path, account_id, api_token):
    url = f"https://api.cloudflare.com/client/v4/accounts/{account_id}/images/v1"
    headers = {"Authorization": f"Bearer {api_token}"}
    
    with open(image_path, 'rb') as f:
        files = {'file': f}
        response = requests.post(url, headers=headers, files=files)
    
    if response.status_code == 200:
        result = response.json()
        return result['result']['variants'][0]
    else:
        return f"Error: {response.text}"

if cloudflare_account_id and cloudflare_api_token:
    # 最新の画像を取得
    images = sorted([f for f in os.listdir(image_path) if f.endswith('.png')], 
                   key=lambda x: os.path.getmtime(os.path.join(image_path, x)), 
                   reverse=True)
    
    if images:
        latest = os.path.join(image_path, images[0])
        print(f"\n📤 アップロード中: {images[0]}")
        result = upload_to_cloudflare(latest, cloudflare_account_id, cloudflare_api_token)
        print(f"\n✅ 完了！\n\nURL: {result}")
    else:
        print("⚠️ 画像が見つかりません")
else:
    print("⚠️ Cloudflare認証情報を入力してください")

In [ ]:
#@title 💾 **Google Driveに保存**
#@markdown 生成物をDriveにバックアップ

from google.colab import drive
drive.mount('/content/drive')

import shutil
import os
from datetime import datetime

# バックアップ先
backup_dir = f"/content/drive/MyDrive/IWASAKI_AI_Studio/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(backup_dir, exist_ok=True)

# Fooocus出力をコピー
if os.path.exists('/content/Fooocus/outputs'):
    shutil.copytree('/content/Fooocus/outputs', f'{backup_dir}/images', dirs_exist_ok=True)
    print(f"✅ 画像を保存: {backup_dir}/images")

# 動画をコピー
for f in os.listdir('/content'):
    if f.endswith('.mp4'):
        shutil.copy(f'/content/{f}', f'{backup_dir}/{f}')
        print(f"✅ 動画を保存: {backup_dir}/{f}")

# LoRAをコピー
if os.path.exists('/content/output_lora'):
    shutil.copytree('/content/output_lora', f'{backup_dir}/lora', dirs_exist_ok=True)
    print(f"✅ LoRAを保存: {backup_dir}/lora")

print(f"\n✅ 全てバックアップ完了！\n📁 {backup_dir}")

---

## 🔥 使い方まとめ

1. **Step 0** を実行してGPU確認
2. やりたいセクションを順番に実行
3. 生成物はGoogle DriveかCloudflareに保存

---

**「どうせうまくいかない」精神で。でも、やる。**

--- 

*IWASAKI AI STUDIO v1.0*
*Built by Human + AI*